# SeedNet: procedural frozen weights

This notebook validates the project from first principles:

1. deterministic seed-to-weight generation;
2. equivalence with an explicit dense matrix;
3. `SeedLinear` with a trainable low-rank correction;
4. a compact training example;
5. storage accounting;
6. optional CUDA/Triton correctness and timing.

The fused kernel avoids persistent storage and HBM reads for the frozen base
matrix. It does not eliminate matrix-multiplication arithmetic or activation
memory.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent]
repo = next((p for p in candidates if (p / "src" / "seednet").exists()), None)
if repo is None:
    raise RuntimeError("Run this notebook from the repository or notebooks directory.")
sys.path.insert(0, str(repo / "src"))

import torch
from seednet import SeedLinear, materialize_seed_weight, triton_available
from seednet.functional import seed_gemm

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Triton backend:", triton_available())

## 1. Determinism and dense equivalence

In [ ]:
seed = 0xBEEF
w1 = materialize_seed_weight(64, 128, seed)
w2 = materialize_seed_weight(64, 128, seed)
assert torch.equal(w1, w2)

x = torch.randn(16, 128)
procedural = seed_gemm(x, 64, seed, backend="reference")
dense = x @ w1.t()
torch.testing.assert_close(procedural, dense)
print("Determinism and dense equivalence: passed")

## 2. SeedLinear and gradient flow

In [ ]:
layer = SeedLinear(128, 64, seed=seed, rank=8, backend="reference")
x = torch.randn(16, 128, requires_grad=True)
y = layer(x)
loss = y.square().mean()
loss.backward()

print("output:", tuple(y.shape))
print("input gradient:", tuple(x.grad.shape))
print("adapter gradient norms:",
      layer.adapter_A.grad.norm().item(),
      layer.adapter_B.grad.norm().item())
print(layer.storage_report())

## 3. Small classification experiment

In [ ]:
torch.manual_seed(0)
features, classes = 64, 5
teacher = torch.randn(classes, features)
x_train = torch.randn(4096, features)
y_train = (x_train @ teacher.t()).argmax(dim=-1)

model = torch.nn.Sequential(
    SeedLinear(features, 256, seed=101, rank=16, backend="reference"),
    torch.nn.GELU(),
    SeedLinear(256, classes, seed=202, rank=8, backend="reference"),
)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(201):
    idx = torch.randint(0, len(x_train), (256,))
    logits = model(x_train[idx])
    loss = torch.nn.functional.cross_entropy(logits, y_train[idx])
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 50 == 0:
        with torch.no_grad():
            acc = (model(x_train[:1024]).argmax(-1) == y_train[:1024]).float().mean()
        print(f"step={step:3d} loss={loss.item():.4f} accuracy={acc.item():.3f}")

## 4. Optional fused CUDA/Triton validation

In [ ]:
if triton_available():
    device = "cuda"
    dtype = torch.float16
    x = torch.randn(65, 70, device=device, dtype=dtype, requires_grad=True)
    seed = 77
    fused = seed_gemm(x, 73, seed, backend="triton")
    w = materialize_seed_weight(73, 70, seed, device=device, dtype=dtype)
    dense = x @ w.t()
    torch.testing.assert_close(fused, dense, rtol=2e-2, atol=2e-2)

    grad = torch.randn_like(fused)
    fused.backward(grad)
    torch.testing.assert_close(x.grad, grad @ w, rtol=2e-2, atol=2e-2)
    print("Fused forward and backward validation: passed")
else:
    print("Skipped: CUDA/Triton backend unavailable.")

## Interpretation checklist

- The seed replaces persistent storage of the frozen base matrix, not the cost of using it.
- The low-rank correction is the principal trainable capacity in this release.
- Large matrices remain expensive in FLOPs even when their weights are procedural.
- Benchmark fused generation against a dense baseline on the actual target shapes.
- Treat the hash/version specification as part of every durable checkpoint.